<div style="display: flex; align-items: center; gap: 20px; background-color: rgba(9, 169, 36, 0.72); padding: 20px; border-radius: 10px; color: white;">
  <img src="logo_udea.png" alt="Logo UdeA" style="height: 80px;">
  <div>
    <h1 style="margin: 0; font-size: 2em;">Proyecto 2 - Análisis espectral de señales ECG</h1>
    <p style="margin: 2px 0 0 0; font-size: 1.3em;"><strong>Bioseñales y Sistemas</strong></p>
    <p style="margin: 10px 0 0 0; font-size: 1.2em;">Universidad de Antioquia - Facultad de Ingeniería</p>
    <p style="margin: 5px 0 0 0; font-size: 1em;">Integrantes: Luisa Taho, Liseth Tiria, Victor Ocampo</p>
    <p style="margin: 5px 0 0 0; font-size: 1em;">Profesores: John Fredy Ochoa, Juliana Moreno Rada</p>
  </div>
</div>



# Importación de librerías y funciones

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy import fft
import matplotlib.pyplot as plt;
import scipy.io as sio;
import seaborn as sns
from scipy.signal import detrend
from scipy.stats import ttest_ind, shapiro, levene, mannwhitneyu
from scipy import stats
import os
import zipfile
from scipy.signal import welch

# Cargue del dataset

### a) Cargue del archivo "Diagnostics.xlsx'

In [3]:
data = pd.read_excel('C:\Programación\Bioseñales\proyecto2_ekg\Diagnostics.xlsx')
data

,FileName,Rhythm,Beat,PatientAge,Gender,VentricularRate,AtrialRate,QRSDuration,QTInterval,QTCorrected,RAxis,TAxis,QRSCount,QOnset,QOffset,TOffset
0,MUSE_20180113_171327_27000,AFIB,RBBB TWC,85,MALE,117,234,114,356,496,81,-27,19,208,265,386
1,MUSE_20180112_073319_29000,SB,TWC,59,FEMALE,52,52,92,432,401,76,42,8,215,261,431
2,MUSE_20180111_165520_97000,SA,NONE,20,FEMALE,67,67,82,382,403,88,20,11,224,265,415
3,MUSE_20180113_121940_44000,SB,NONE,66,MALE,53,53,96,456,427,34,3,9,219,267,447
4,MUSE_20180112_122850_57000,AF,STDD STTC,73,FEMALE,162,162,114,252,413,68,-40,26,228,285,354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10641,MUSE_20181222_204306_99000,SVT,NONE,80,FEMALE,196,73,168,284,513,258,244,32,177,261,319
10642,MUSE_20181222_204309_22000,SVT,NONE,81,FEMALE,162,81,162,294,482,110,-75,27,173,254,320
10643,MUSE_20181222_204310_31000,SVT,NONE,39,MALE,152,92,152,340,540,250,38,25,208,284,378
10644,MUSE_20181222_204312_58000,SVT,NONE,76,MALE,175,178,128,310,529,98,-83,29,205,269,360


In [33]:
dataSB_FileName = data.loc[lambda x: x["Rhythm"] == "SB", "FileName"].tolist()
dataAFIB_FileName = data.loc[lambda x: x["Rhythm"] == "AFIB", "FileName"].tolist()
dataSR_FileName = data.loc[lambda x: x["Rhythm"] == "SR", "FileName"].tolist()

print(f"Cantidad de archivos para SB (bradicardia sinusal): {len(dataSB_FileName)}")
print(f"Cantidad de archivos para AFIB (fibrilación auricular): {len(dataAFIB_FileName)}")
print(f"Cantidad de archivos para SR (ritmo sinusal): {len(dataSR_FileName)}")

Cantidad de archivos para SB (bradicardia sinusal): 3889
Cantidad de archivos para AFIB (fibrilación auricular): 1780
Cantidad de archivos para SR (ritmo sinusal): 1826


In [34]:

dataAFIB_FileName_csv = [f"{archivo}.csv" for archivo in dataAFIB_FileName]
dataSB_FileName_csv = [f"{archivo}.csv" for archivo in dataSB_FileName]
dataSR_FileName_csv = [f"{archivo}.csv" for archivo in dataSR_FileName]

In [29]:
directorio_actual = os.getcwd()
folder = os.listdir(directorio_actual+"/ECGDataDenoised/ECGDataDenoised")

In [35]:
SB_registers = []
AFIB_registers = []
SR_registers = []

for archivo in folder:
    if archivo in dataSB_FileName_csv:
        SB_registers.append(archivo)
    if archivo in dataAFIB_FileName_csv:
        AFIB_registers.append(archivo)
    if archivo in dataSR_FileName_csv:
        SR_registers.append(archivo)
    




In [36]:
len(SB_registers), len(AFIB_registers), len(SR_registers)

(3889, 1780, 1826)

# Extracción de características 

### Extracción de características usando la Transformada Discreta de Fourier